In [1]:
import os
from pyflink.table import EnvironmentSettings, TableEnvironment, TableDescriptor, Schema, DataTypes

In [2]:
table_environment = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
configration = table_environment.get_config().get_configuration()
configration.set_string("execution.target", "remote")
configration.set_string("rest.address", "flink-jobmanager")
configration.set_integer("rest.port", 8082)
configration.set_string("execution.checkpointing.interval", "3 s")
configration.set_string("execution.checkpointing.mode", "EXACTLY_ONCE")
configration.set_string("execution.checkpointing.timeout", "10 min")
#configration.set_string("pipeline.jars","file:///ABS/PATH/flink-sql-avro-confluent-registry-1.20.3.jar")

## Mysql -> Debezium -> Kafka -> Flink Source Table

In [3]:
table_environment.execute_sql("CREATE DATABASE IF NOT EXISTS mmix_mysql_cdc;")
table_environment.execute_sql("USE mmix_mysql_cdc;")

In [4]:
# kafka_source_descriptor = (
#     TableDescriptor
#     .for_connector("kafka")
#     .schema(Schema.new_builder()
#             .column("id", DataTypes.INT())
#             .column("name", DataTypes.STRING())
#             .column("department", DataTypes.INT())
#             .column("salary", DataTypes.DOUBLE())
#             .build())
#     .option("topic", "debezium.mysql.source.mmix.employees")
#     .option("properties.bootstrap.servers", "confluent-kafka-broker:19092")
#     .option("properties.group.id", "mmix-debezium-mysql-source-flink")
#     .option("scan.startup.mode", "earliest-offset")
#     .option("format", "avro-confluent")
#     .option("avro-confluent.schema-registry.url", "http://confluent-schema-registry:8081")
#     .build())
# table_environment.create_table("mmix_employees_kafka_source", kafka_source_descriptor)
# result = table_environment.execute_sql("SELECT id, name, department, salary FROM mmix_employees_kafka_source")
table_environment.execute_sql("""
CREATE TABLE mmix_employees_kafka_source
(
  id         INT,
  name       STRING,
  department INT,
  salary DOUBLE
)
  WITH (
      'connector' = 'kafka',
      'topic' = 'debezium.mysql.source.mmix.employees',
      'properties.bootstrap.servers' = 'confluent-kafka-broker:9092',
      'properties.group.id' = 'mmix-debezium-mysql-source-flink',
      'scan.startup.mode' = 'earliest-offset',
      'format' = 'avro-confluent',
      'avro-confluent.schema-registry.url' = 'http://confluent-schema-registry:8081'
      );
""")

In [5]:
table_environment.execute_sql("""
CREATE TABLE mmix_employees_iceberg_sink (
      id          INT,
      name        STRING,
      department  INT,
      salary      DOUBLE,
      PRIMARY KEY (id) NOT ENFORCED
) WITH (
  'connector' = 'iceberg',
  'catalog-name' = 'iceberg_catalog',
  'catalog-type' = 'hadoop',
  'warehouse' = 's3a://mmix-prod-dataengineer-datalakehouse/streaming',
  'format-version' = '2'
);
""")

In [6]:
table_environment.execute_sql("INSERT INTO mmix_employees_iceberg_sink SELECT id, name, department, salary FROM mmix_employees_kafka_source;").print()

2026-01-17 19:46:16,300 WARN  org.apache.hadoop.metrics2.impl.MetricsConfig                [] - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
2026-01-17 19:46:16,352 INFO  org.apache.hadoop.metrics2.impl.MetricsSystemImpl            [] - Scheduled Metric snapshot period at 10 second(s).
2026-01-17 19:46:16,352 INFO  org.apache.hadoop.metrics2.impl.MetricsSystemImpl            [] - s3a-file-system metrics system started


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1217, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 